# 00 - Data, history features, leakage audit and frozen splits (`nhanes_v003`)

Run order: **00** -> 01 EBM -> 02 CatBoost -> 03 XGBoost -> 04 LightGBM -> 04b logistic-regression reference -> 05 compare and register.

This notebook
1. re-verifies the cohort CSV against the raw `KIQ_U` files (row and positive counts per cycle),
2. builds the **v2 cohort** (self-reported history features added, `KIQ029` dropped),
3. checks every history variable per survey cycle and keeps only those that are present with the same meaning in the train, validation and test cycles,
4. runs a single-feature AUC **leakage audit**,
5. **freezes** the cycle-based train / validation / test row IDs with a hash, so every model notebook provably uses the same split,
6. screens the candidate feature sets on **train + validation only** (the test cycle is never scored here) and writes `feature_spec.json`.

The test cycle (2017-2018) is only counted (rows, prevalence); no model sees it until the last section of each model notebook.

## 1. Setup

In [1]:
import sys, json, time, warnings
from pathlib import Path

ROOT = next(p for p in [Path.cwd().resolve(), *Path.cwd().resolve().parents]
            if (p / "ml" / "training" / "nhanes_common.py").exists())
sys.path.insert(0, str(ROOT))

import numpy as np
import pandas as pd
warnings.filterwarnings("ignore")

from ml.training import nhanes_common as nc

nc.set_seeds()
pd.set_option("display.width", 150)
pd.set_option("display.max_columns", 40)
print("repo root:", ROOT.name)
print("dry run (test cycle untouched, outputs in ml/outputs/_dryrun):", nc.is_dry_run())

repo root: StoneSense-AI
dry run (test cycle untouched, outputs in ml/outputs/_dryrun): False


## 2. Re-verify the cohort against the raw KIQ_U files

The cohort keeps participants whose `KIQ026` answer ("Ever had kidney stones?") is Yes (1) or No (2); refused / don't-know / missing answers are dropped. For every cycle the cohort row count must equal the number of raw rows with a usable answer, and the positives must equal the raw "Yes" count.

In [2]:
from ml.preprocessing import add_history_features as ahf

raw_dir = nc.datasets_dir() / "nhanes_raw"
cohort = pd.read_csv(ahf.COHORT_PATH)
rows = []
for cycle, sfx in ahf.CYCLE_SUFFIX.items():
    k = pd.read_sas(raw_dir / f"KIQ_U_{sfx}.xpt", format="xport")
    c = cohort[cohort["CYCLE"] == cycle]
    rows.append({
        "cycle": cycle, "raw_rows": len(k), "raw_usable_answer_rows": int(k["KIQ026"].isin([1.0, 2.0]).sum()),
        "raw_yes": int((k["KIQ026"] == 1).sum()), "cohort_rows": len(c), "cohort_positives": int(c["target_stone"].sum()),
    })
cohort_check = pd.DataFrame(rows)
cohort_check["rows_match"] = cohort_check["raw_usable_answer_rows"] == cohort_check["cohort_rows"]
cohort_check["positives_match"] = cohort_check["raw_yes"] == cohort_check["cohort_positives"]
display(cohort_check)
assert cohort_check["rows_match"].all() and cohort_check["positives_match"].all()
print("Cohort:", len(cohort), "rows,", int(cohort["target_stone"].sum()), "positives (",
      round(100 * cohort["target_stone"].mean(), 2), "% ); SEQN unique:", cohort["SEQN"].is_unique)

,cycle,raw_rows,raw_usable_answer_rows,raw_yes,cohort_rows,cohort_positives,rows_match,positives_match
0,2007-2008,5935,5909,539,5909,539,True,True
1,2009-2010,6218,6201,542,6201,542,True,True
2,2011-2012,5560,5548,458,5548,458,True,True
3,2013-2014,5769,5758,535,5758,535,True,True
4,2015-2016,5719,5705,606,5705,606,True,True
5,2017-2018,5569,5558,554,5558,554,True,True


Cohort: 34679 rows, 3234 positives ( 9.33 % ); SEQN unique: True


## 3. Build the v2 cohort and check each history variable per cycle

`BPQ020, BPQ080, DIQ010, MCQ080, MCQ520, MCQ550` are read from the 18 history files. Codes 7 (refused) and 9 (don't know) become missing, yes/no becomes 1/0 (`DIQ010`: borderline = 0.5). The merge is a one-to-one left merge on `SEQN`; row and label counts are asserted unchanged and **`KIQ029` is dropped**. The original cohort CSV is never overwritten.

A history feature may be used only if it is **present, with the same question label, in every train, validation and test cycle**. A whole missing cycle is never imputed.

In [3]:
v2, avail = ahf.build_v2_cohort(cohort)
assert len(v2) == len(cohort) and v2["target_stone"].sum() == cohort["target_stone"].sum() and "KIQ029" not in v2.columns
v2.to_csv(ahf.COHORT_V2_PATH, index=False)
avail.to_csv(ahf.AVAILABILITY_PATH, index=False)
print("v2 cohort:", v2.shape, "-> ", ahf.COHORT_V2_PATH.name)
show = avail.assign(percent_missing=np.where(avail["present"], avail["percent_missing"].astype(str), "absent"))
display(show.pivot(index="variable", columns="cycle", values="percent_missing"))
print("(values = percent missing after decoding 7/9 -> NaN; 'absent' = variable not collected in that cycle)")

v2 cohort: (34679, 33) ->  nhanes_kidney_stone_cohort_v2.csv


cycle,2007-2008,2009-2010,2011-2012,2013-2014,2015-2016,2017-2018
variable,,,,,,
BPQ020,0.1,0.18,0.14,0.07,0.11,0.16
BPQ080,29.55,30.43,0.63,0.57,0.75,0.67
DIQ010,0.1,0.06,0.04,0.05,0.04,0.07
MCQ080,0.1,0.06,0.05,0.02,0.07,0.11
MCQ520,absent,absent,absent,absent,absent,0.09
MCQ550,absent,absent,absent,absent,absent,0.27


(values = percent missing after decoding 7/9 -> NaN; 'absent' = variable not collected in that cycle)


In [4]:
decisions = ahf.select_history_features(avail, nc.TRAIN_CYCLES, nc.VAL_CYCLES, nc.TEST_CYCLES)
usable_history = [v for v, d in decisions.items() if d["usable"]]
for v, d in decisions.items():
    print(f"{v}: {'USE    ' if d['usable'] else 'EXCLUDE'} - {d['reason']}")
print("\nUsable history features:", usable_history)
history_questions = {v: ahf.HISTORY_VARS[v][1] for v in usable_history}
bpq = avail[avail.variable == "BPQ080"].set_index("cycle")["percent_missing"]
print(f"\nNote BPQ080 missingness: {bpq.loc[nc.TRAIN_CYCLES].round(1).tolist()} in the training cycles vs "
      f"{bpq.loc[nc.VAL_CYCLES + nc.TEST_CYCLES].round(1).tolist()} in validation/test (older cycles skipped the cholesterol "
      "question for people never screened). The question is the same; only the amount of missing differs, and it is imputed per row, never per cycle.")

BPQ020: USE     - present with identical label in all train/validation/test cycles
BPQ080: USE     - present with identical label in all train/validation/test cycles
DIQ010: USE     - present with identical label in all train/validation/test cycles
MCQ080: USE     - present with identical label in all train/validation/test cycles
MCQ520: EXCLUDE - not collected in cycle(s) 2007-2008, 2009-2010, 2011-2012, 2013-2014, 2015-2016
MCQ550: EXCLUDE - not collected in cycle(s) 2007-2008, 2009-2010, 2011-2012, 2013-2014, 2015-2016

Usable history features: ['BPQ020', 'BPQ080', 'DIQ010', 'MCQ080']

Note BPQ080 missingness: [29.6, 30.4, 0.6, 0.6] in the training cycles vs [0.8, 0.7] in validation/test (older cycles skipped the cholesterol question for people never screened). The question is the same; only the amount of missing differs, and it is imputed per row, never per cycle.


## 4. Single-feature leakage audit

Each candidate feature is scored alone (ROC-AUC, direction-agnostic) on the **training cycles**. Anything above 0.85 would be flagged as a possible label proxy. The identifiers, the label and its proxies (`SEQN, CYCLE, target_stone, KIQ026, KIQ029`) and the survey-design columns (`WTMEC2YR, WTINT2YR, SDMVPSU, SDMVSTRA`) are **never** features; this is asserted below.

In [5]:
from ml.risk_models.nhanes_bundle import BANNED_COLUMNS

variants = nc.make_variants(usable_history)       # asserts no banned column in any variant
train_rows = v2[v2["CYCLE"].isin(nc.TRAIN_CYCLES)]
candidates = nc.BASE_FEATURES + usable_history + nc.SENSITIVE_FEATURES
audit = nc.leakage_audit(train_rows, candidates)
display(audit)
print("Banned columns (never features):", ", ".join(BANNED_COLUMNS))
print("Max single-feature AUC:", audit["single_feature_auc"].max(), "| flagged:", audit.loc[audit.flagged, "feature"].tolist())
assert not audit["flagged"].any()

,feature,single_feature_auc,flagged
0,RIDAGEYR,0.6164,False
1,BMXWAIST,0.6010,False
2,BPQ020,0.5850,False
3,LBXSBU,0.5810,False
4,URXUMA,0.5771,False
5,LBXSGL,0.5715,False
6,BMXBMI,0.5705,False
7,BPQ080,0.5693,False
8,MCQ080,0.5621,False
9,LBXSCR,0.5605,False


Banned columns (never features): SEQN, CYCLE, target_stone, KIQ026, KIQ029, WTMEC2YR, WTINT2YR, SDMVPSU, SDMVSTRA
Max single-feature AUC: 0.6164 | flagged: []


## 5. Freeze the cycle-based splits

* train = 2007-2008, 2009-2010, 2011-2012, 2013-2014
* validation = 2015-2016 (early stopping, tuning, calibration, threshold selection)
* test = 2017-2018 (used exactly once per model, at the very end of each model notebook)

No random split. The row IDs are saved to `ml/datasets/nhanes_v003/split_ids.json` (git-ignored data) and a hash is stored in `ml/artifacts/nhanes_v003/split_manifest.json`; every model notebook re-checks the hash on load. If prevalence differs between splits this is reported here, not "fixed" by changing the split.

In [6]:
manifest = nc.freeze_splits(v2)
all_splits = nc.split_by_cycle(v2)
summary = nc.split_summary(all_splits)
display(summary)
print("split hash:", manifest["hash"][:16], "...  cohort sha256:", manifest["cohort_sha256"][:16], "...")
assert not set(all_splits["train"].SEQN) & set(all_splits["validation"].SEQN)
assert not set(all_splits["train"].SEQN) & set(all_splits["test"].SEQN)
assert not set(all_splits["validation"].SEQN) & set(all_splits["test"].SEQN)
print("No SEQN is shared between train, validation and test; cycles are disjoint.")
rate = summary.set_index("split")["positive_rate_pct"]
print(f"Positive rate: train {rate['train']}%  validation {rate['validation']}%  test {rate['test']}%")
if (rate - rate["train"]).abs().max() > 1.0:
    print("NOTE: prevalence differs by more than 1 percentage point between splits (validation/test are above train). "
          "Reported as-is; the split was not adjusted.")
print("Majority-class ('no stone for everyone') accuracy: validation",
      round(100 - rate['validation'], 2), "%, test", round(100 - rate['test'], 2), "%")

,split,cycles,rows,positives,positive_rate_pct
0,train,"2007-2008, 2009-2010, 2011-2012, 2013-2014",23416,2074,8.86
1,validation,2015-2016,5705,606,10.62
2,test,2017-2018,5558,554,9.97


split hash: a2c3bdb15ffb9f3b ...  cohort sha256: 4b6cf9a2e01ed265 ...
No SEQN is shared between train, validation and test; cycles are disjoint.
Positive rate: train 8.86%  validation 10.62%  test 9.97%
NOTE: prevalence differs by more than 1 percentage point between splits (validation/test are above train). Reported as-is; the split was not adjusted.
Majority-class ('no stone for everyone') accuracy: validation 89.38 %, test 90.03 %


## 6. Screen the feature sets (train + validation only)

Three variants, all trained later in every model notebook:

| variant | contents |
|---|---|
| `base` | the 13 `xgboost_nhanes_v002` features + `BMXWAIST, LBXSCLSI, LBXSTB, URDACT` |
| `base_history` | `base` + the history features that passed the per-cycle check |
| `base_history_sensitive` | the above + race/ethnicity (`RIDRETH1`) and income ratio (`INDFMPIR`) |

Race and income are ethically sensitive inputs for a clinical tool, so they are **off by default** and adopted only if they *clearly* help. A small, fixed LightGBM (CPU) is used as the screening model. Metrics: validation ROC-AUC, leave-one-cycle-out ROC-AUC over the four training cycles, and a grouped CV robustness check (group = cycle + `SDMVSTRA` + `SDMVPSU`) over train+validation.

Rule (written before looking at the numbers):
* adopt history if mean LOCO AUC improves by >= 0.005 **and** validation AUC does not drop;
* adopt the sensitive variables only if, on top of that, mean LOCO AUC **and** validation AUC both improve by >= 0.010.

In [7]:
import lightgbm as lgb
from sklearn.metrics import roc_auc_score

def screen_fit_predict(Xtr, ytr, Xte):
    m = lgb.LGBMClassifier(n_estimators=250, learning_rate=0.03, num_leaves=8, min_child_samples=100, subsample=0.7,
                           subsample_freq=1, colsample_bytree=0.7, reg_lambda=5.0, scale_pos_weight=nc.class_weight_ratio(ytr),
                           random_state=nc.SEED, n_jobs=4, verbose=-1)
    return m.fit(Xtr, ytr).predict_proba(Xte)[:, 1]

splits = nc.load_frozen_splits(include_test=False)       # train + validation only
dev = pd.concat([splits["train"], splits["validation"]], ignore_index=True)
screen_rows = []
for name, feats in variants.items():
    vd = nc.prepare_variant(splits, name, feats)
    val_auc = roc_auc_score(vd.y_val, screen_fit_predict(vd.X_train, vd.y_train, vd.X_val))
    loco = nc.loco_cv(splits["train"], feats, screen_fit_predict, nc.TRAIN_CYCLES)
    grp = nc.grouped_cv(dev, feats, screen_fit_predict, n_splits=5)
    screen_rows.append({"variant": name, "n_features": len(feats), "val_auc": round(val_auc, 4),
                        "loco_auc_mean": round(loco["roc_auc_mean"], 4), "loco_auc_std": round(loco["roc_auc_std"], 4),
                        "loco_auc_min": round(loco["roc_auc_min"], 4), "loco_auc_max": round(loco["roc_auc_max"], 4),
                        "grouped_cv_auc_mean": round(grp["roc_auc_mean"], 4), "grouped_cv_auc_std": round(grp["roc_auc_std"], 4)})
screen = pd.DataFrame(screen_rows).set_index("variant")
display(screen)

,n_features,val_auc,loco_auc_mean,loco_auc_std,loco_auc_min,loco_auc_max,grouped_cv_auc_mean,grouped_cv_auc_std
variant,,,,,,,,
base,17,0.6350,0.6513,0.0030,0.6483,0.6554,0.6500,0.0143
base_history,21,0.6433,0.6613,0.0010,0.6604,0.6622,0.6601,0.0163
base_history_sensitive,23,0.6616,0.6848,0.0032,0.6820,0.6884,0.6813,0.0076


In [8]:
import os
s = screen
use_history = bool(usable_history) and (s.loc["base_history", "loco_auc_mean"] - s.loc["base", "loco_auc_mean"] >= 0.005) \
    and (s.loc["base_history", "val_auc"] >= s.loc["base", "val_auc"])
primary = "base_history" if use_history else "base"
use_sensitive = use_history and (s.loc["base_history_sensitive", "loco_auc_mean"] - s.loc["base_history", "loco_auc_mean"] >= 0.010) \
    and (s.loc["base_history_sensitive", "val_auc"] - s.loc["base_history", "val_auc"] >= 0.010)
if use_sensitive:
    primary = "base_history_sensitive"
override = os.environ.get("NHANES_PRIMARY_VARIANT")      # explicit, documented override (e.g. clinical decision)
if override:
    assert override in variants, override
    primary = override
print("history adopted:", use_history, "| sensitive adopted:", use_sensitive, "| override:", override)
print("PRIMARY VARIANT (registered, shown in the app):", primary)

cfg = {
    "primary_variant": primary, "variants": variants, "base_features": nc.BASE_FEATURES, "sensitive_features": nc.SENSITIVE_FEATURES,
    "history_features_used": usable_history, "history_questions": history_questions,
    "history_excluded": {v: d["reason"] for v, d in decisions.items() if not d["usable"]},
    "screening": screen.reset_index().to_dict(orient="records"),
    "selection_rule": "history if LOCO-AUC +0.005 and val-AUC not lower; sensitive only if LOCO-AUC and val-AUC both +0.010",
    "split_hash": manifest["hash"], "seed": nc.SEED,
}
nc.save_json(cfg, nc.artifacts_dir() / "feature_config.json")
spec = nc.build_feature_spec(cfg, splits["train"])
nc.save_json(spec, nc.artifacts_dir() / "feature_spec.json")
print("\nfeature_spec.json fields for the registered variant:")
display(pd.DataFrame(spec["features"])[["name", "label", "unit", "type", "min", "max", "default", "required", "group"]])

history adopted: True | sensitive adopted: True | override: None
PRIMARY VARIANT (registered, shown in the app): base_history_sensitive

feature_spec.json fields for the registered variant:


,name,label,unit,type,min,max,default,required,group
0,RIDAGEYR,Age,years,continuous,20.0,80.0,49.00,True,Demographics
1,RIAGENDR,Sex,,binary,1.0,2.0,2.00,True,Demographics
2,BMXBMI,Body mass index,kg/m2,continuous,10.0,90.0,27.52,True,Body measures
3,LBXSCA,Serum calcium,mg/dL,continuous,5.0,16.0,9.40,True,Blood chemistry
4,LBXSUA,Serum uric acid,mg/dL,continuous,0.5,20.0,5.20,True,Blood chemistry
5,LBXSCR,Serum creatinine,mg/dL,continuous,0.1,20.0,0.82,True,Blood chemistry
6,LBXSBU,Blood urea nitrogen,mg/dL,continuous,1.0,120.0,12.00,True,Blood chemistry
7,LBXSGL,Serum glucose,mg/dL,continuous,20.0,800.0,91.00,True,Blood chemistry
8,LBXSNASI,Serum sodium,mmol/L,continuous,100.0,170.0,139.00,True,Blood chemistry
9,LBXSKSI,Serum potassium,mmol/L,continuous,1.5,8.5,3.90,True,Blood chemistry


In [9]:
nc.save_json({
    "cohort_check": cohort_check.to_dict(orient="records"),
    "history_availability": avail.to_dict(orient="records"),
    "history_decisions": decisions,
    "leakage_audit": audit.to_dict(orient="records"),
    "split_summary": summary.to_dict(orient="records"),
    "split_manifest": manifest,
    "screening": screen.reset_index().to_dict(orient="records"),
    "primary_variant": primary,
}, nc.metrics_dir() / "data_and_splits.json")
print("saved:", nc.rel(nc.metrics_dir() / "data_and_splits.json"))
print("saved:", nc.rel(nc.artifacts_dir() / "feature_spec.json"))
print("saved:", nc.rel(nc.artifacts_dir() / "split_manifest.json"))

saved: ml/outputs/metrics/nhanes_v003/data_and_splits.json
saved: ml/artifacts/nhanes_v003/feature_spec.json
saved: ml/artifacts/nhanes_v003/split_manifest.json
